In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

In [ ]:
#teiling 640x640 with 200 overlap
#경계 부분 zero padding

def load_yolo_labels(label_path, img_w, img_h):
    labels = []
    with open(label_path, "r") as f:
        for line in f.readlines():
            cls, cx, cy, w, h = map(float, line.split())

            # YOLO normalized → absolute
            x1 = (cx - w/2) * img_w
            y1 = (cy - h/2) * img_h
            x2 = (cx + w/2) * img_w
            y2 = (cy + h/2) * img_h

            labels.append([cls, x1, y1, x2, y2])
    return labels


def tile_image_and_labels(img, labels, save_dir, base_name,
                          tile_size=640, overlap=200):
    
    h, w = img.shape[:2]
    ts = tile_size
    ov = overlap

    tile_id = 0

    for ty in range(0, h, ts - ov):
        for tx in range(0, w, ts - ov):

            # 타일 좌표가 이미지 범위를 벗어날 수 있으므로 padding
            tile = np.zeros((ts, ts, 3), dtype=np.uint8)

            real_tile = img[ty:ty+ts, tx:tx+ts]
            rh, rw = real_tile.shape[:2]
            tile[:rh, :rw] = real_tile

            # tile 내 bbox 리스트
            tile_labels = []

            for cls, x1, y1, x2, y2 in labels:
                # 타일과 안 겹침 → skip
                if x2 < tx or x1 > tx + ts or y2 < ty or y1 > ty + ts:
                    continue

                # 타일 기준 crop
                nx1 = max(0, x1 - tx)
                ny1 = max(0, y1 - ty)
                nx2 = min(ts, x2 - tx)
                ny2 = min(ts, y2 - ty)

                bw = nx2 - nx1
                bh = ny2 - ny1

                if bw <= 1 or bh <= 1:
                    continue

                # YOLO 재정규화
                ncx = (nx1 + nx2) / 2 / ts
                ncy = (ny1 + ny2) / 2 / ts
                nbw = bw / ts
                nbh = bh / ts

                tile_labels.append([cls, ncx, ncy, nbw, nbh])

            # 저장
            tile_name = f"{base_name}_{tile_id}"
            cv2.imwrite(os.path.join(save_dir, "images", tile_name + ".jpg"), tile)

            with open(os.path.join(save_dir, "labels", tile_name + ".txt"), "w") as f:
                for t in tile_labels:
                    f.write(" ".join(map(str, t)) + "\n")

            tile_id += 1


def make_tiled_dataset(img_dir, label_dir, out_dir,
                       tile_size=640, overlap=200):

    os.makedirs(os.path.join(out_dir, "images"), exist_ok=True)
    os.makedirs(os.path.join(out_dir, "labels"), exist_ok=True)

    img_files = [f for f in os.listdir(img_dir)
                 if f.lower().endswith((".jpg", ".png", ".jpeg"))]

    for img_name in tqdm(img_files):
        base = os.path.splitext(img_name)[0]

        img_path = os.path.join(img_dir, img_name)
        label_path = os.path.join(label_dir, base + ".txt")

        img = cv2.imread(img_path)
        h, w = img.shape[:2]

        labels = load_yolo_labels(label_path, w, h)

        tile_image_and_labels(img, labels, out_dir, base,
                              tile_size, overlap)



In [ ]:
make_tiled_dataset(
    img_dir=r"C:\Users\lijih\code\drone.v4i.yolov8\valid\images",
    label_dir=r"C:\Users\lijih\code\drone.v4i.yolov8\valid\labels",
    out_dir=r"C:\Users\lijih\code\drone_dataset\valid",
    tile_size=640,
    overlap=200
)

In [ ]:
#이미지 크기에 맞게 타일 크기 조정
def load_yolo_labels(label_path, img_w, img_h):
    labels = []
    if not os.path.exists(label_path):
        return labels

    with open(label_path, 'r') as f:
        for line in f:
            if line.strip() == "":
                continue

            cls, cx, cy, w, h = map(float, line.split())
            x1 = (cx - w/2) * img_w
            y1 = (cy - h/2) * img_h
            x2 = (cx + w/2) * img_w
            y2 = (cy + h/2) * img_h
            labels.append([cls, x1, y1, x2, y2])

    return labels


def tile_image_and_labels(img, labels, save_dir, base_name,
                          tiles_x, tiles_y, overlap_x, overlap_y):

    H, W = img.shape[:2]

    # 비율 기반 타일 크기 자동 계산 (오버랩 포함)
    tile_w = int((W + (tiles_x - 1) * overlap_x) / tiles_x)
    tile_h = int((H + (tiles_y - 1) * overlap_y) / tiles_y)

    # stride = 타일 간 이동 거리
    stride_x = tile_w - overlap_x
    stride_y = tile_h - overlap_y

    tile_id = 0

    for yi in range(tiles_y):
        for xi in range(tiles_x):

            x_start = int(xi * stride_x)
            y_start = int(yi * stride_y)
            x_end = x_start + tile_w
            y_end = y_start + tile_h

            # 타일 crop 후 필요시 padding
            pad_right = max(0, x_end - W)
            pad_bottom = max(0, y_end - H)

            tile = img[y_start:min(y_end, H), x_start:min(x_end, W)]

            if pad_right > 0 or pad_bottom > 0:
                tile = cv2.copyMakeBorder(
                    tile, 0, pad_bottom, 0, pad_right,
                    cv2.BORDER_CONSTANT, value=[0, 0, 0]
                )

            th, tw = tile.shape[:2]
            tile_labels = []

            # bbox 조정
            for cls, x1, y1, x2, y2 in labels:

                if x2 < x_start or x1 > x_end or y2 < y_start or y1 > y_end:
                    continue

                nx1 = max(0, x1 - x_start)
                ny1 = max(0, y1 - y_start)
                nx2 = min(tw, x2 - x_start)
                ny2 = min(th, y2 - y_start)

                if nx2 - nx1 < 2 or ny2 - ny1 < 2:
                    continue

                cx = (nx1 + nx2) / 2 / tw
                cy = (ny1 + ny2) / 2 / th
                bw = (nx2 - nx1) / tw
                bh = (ny2 - ny1) / th

                tile_labels.append([cls, cx, cy, bw, bh])

            tile_name = f"{base_name}_{tile_id}"

            cv2.imwrite(os.path.join(save_dir, "images", tile_name + ".jpg"), tile)

            with open(os.path.join(save_dir, "labels", tile_name + ".txt"), "w") as f:
                for t in tile_labels:
                    f.write(" ".join(map(str, t)) + "\n")

            tile_id += 1


def make_tiled_dataset(img_dir, label_dir, out_dir,
                       tiles_x=3, tiles_y=3,
                       overlap_x=100, overlap_y=100):

    os.makedirs(os.path.join(out_dir, "images"), exist_ok=True)
    os.makedirs(os.path.join(out_dir, "labels"), exist_ok=True)

    img_files = [f for f in os.listdir(img_dir)
                 if f.lower().endswith((".jpg", ".png", ".jpeg"))]

    for img_name in tqdm(img_files):
        base = os.path.splitext(img_name)[0]

        img_path = os.path.join(img_dir, img_name)
        label_path = os.path.join(label_dir, base + ".txt")

        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]
        labels = load_yolo_labels(label_path, w, h)

        tile_image_and_labels(img, labels, out_dir,
                              base, tiles_x, tiles_y,
                              overlap_x, overlap_y)


In [8]:
make_tiled_dataset(
    img_dir=r"C:\Users\lijih\code\drone.v4i.yolov8\train\images",
    label_dir=r"C:\Users\lijih\code\drone.v4i.yolov8\train\labels",
    out_dir=r"C:\Users\lijih\code\drone_dataset_size\train",
    tiles_x=4,   
    tiles_y=3,
    overlap_x=200,
    overlap_y=200
)


100%|██████████| 155/155 [00:37<00:00,  4.12it/s]
